In [49]:
!pip install -q langgraph langchain-core langchain-openai
!pip install -q PyMuPDF Pillow pytesseract requests beautifulsoup4
!pip install -q gradio transformers sentence-transformers
!pip install -q chromadb faiss-cpu pandas numpy

In [50]:
import os
import re
import json
import hashlib
import base64
import requests
from datetime import datetime, timedelta
from typing import Dict, Any, List, Optional, TypedDict
import urllib.parse
from pathlib import Path
import tempfile

# File processing imports
import fitz  # PyMuPDF
from PIL import Image
import pytesseract
import io

# AI/ML imports
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings
import pandas as pd
import numpy as np

# LangGraph imports
from langgraph.graph import END, StateGraph
from langgraph.graph.graph import CompiledGraph

# Web scraping
from bs4 import BeautifulSoup
import ssl
import urllib3

# Gradio for UI
import gradio as gr

# Disable SSL warnings for demo purposes
urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

In [51]:
# =============================================================================
# STATE DEFINITION
# =============================================================================

class VerificationState(TypedDict):
    """State structure for the verification workflow"""
    input_file_path: str
    extracted_text: str
    certifications: List[Dict[str, Any]]
    verification_results: List[Dict[str, Any]]
    credibility_scores: Dict[str, Any]
    flagged_items: List[Dict[str, Any]]
    recommendations: List[str]
    final_report: Dict[str, Any]

In [52]:
# =============================================================================
# AGENT 1: CERTIFICATION EXTRACTION AGENT
# =============================================================================

class CertificationExtractionAgent:
    """Agent responsible for extracting certifications from various document types"""

    def __init__(self):
        self.certification_patterns = {
            'certification_keywords': [
                r'certif(?:ication|ied)',
                r'credential',
                r'license',
                r'course completion',
                r'professional development',
                r'training',
                r'bootcamp',
                r'specialization'
            ],
            'platform_patterns': {
                'Coursera': r'coursera\.[a-z]+/verify/[A-Z0-9]+',
                'edX': r'edx\.org/.*certificate',
                'Udemy': r'udemy\.com/certificate/[A-Z0-9]+',
                'Google': r'google\.com/.*certif',
                'AWS': r'aws\.amazon\.com/.*certif',
                'Microsoft': r'microsoft\.com/.*certif',
                'LinkedIn': r'linkedin\.com/.*certif',
                'Credly': r'credly\.com/badges/[a-z0-9-]+',
                'HackerRank': r'hackerrank\.com/certificates/[a-z0-9]+',
                'FreeCodeCamp': r'freecodecamp\.org/certification/[a-z0-9-]+/[a-z0-9-]+',
                'Pluralsight': r'pluralsight\.com/.*certif',
                'Udacity': r'udacity\.com/certificate/[A-Z0-9]+',
                'IBM': r'ibm\.com/.*certif|ibm\.com/badges',
                'Oracle': r'oracle\.com/.*certif',
                'Cisco': r'cisco\.com/.*certif',
                'CompTIA': r'comptia\.org/.*certif'
            }
        }

        # Initialize sentence transformer for semantic matching
        self.sentence_model = SentenceTransformer('all-MiniLM-L6-v2')

        # Common certification names for matching
        self.common_certifications = [
            "Google Cloud Professional", "AWS Certified", "Microsoft Azure",
            "Python", "Data Science", "Machine Learning", "Full Stack",
            "Cybersecurity", "Project Management", "Digital Marketing",
            "TensorFlow", "React", "Node.js", "Java", "C++", "SQL",
            "Tableau", "Power BI", "Salesforce", "Scrum Master"
        ]

    def extract_from_pdf(self, file_path: str) -> str:
        """Extract text from PDF file"""
        try:
            doc = fitz.open(file_path)
            text = ""
            for page in doc:
                text += page.get_text()
            doc.close()
            print(text)
            return text
        except Exception as e:
            print(f"Error extracting PDF: {e}")
            return ""

    def extract_from_image(self, file_path: str) -> str:
        """Extract text from image using OCR"""
        try:
            image = Image.open(file_path)
            text = pytesseract.image_to_string(image)
            return text
        except Exception as e:
            print(f"Error extracting from image: {e}")
            return ""

    def extract_text_from_file(self, file_path: str) -> str:
        """Extract text from uploaded file"""
        file_ext = Path(file_path).suffix.lower()

        if file_ext == '.pdf':
            return self.extract_from_pdf(file_path)
        elif file_ext in ['.jpg', '.jpeg', '.png', '.tiff', '.bmp']:
            return self.extract_from_image(file_path)
        elif file_ext == '.txt':
            with open(file_path, 'r', encoding='utf-8') as f:
                return f.read()
        else:
            return ""

    def extract_urls(self, text: str) -> List[str]:
        """Extract URLs from text"""
        url_pattern = r'https?://[^\s<>"{}|\\^`\[\]()]+'
        urls = re.findall(url_pattern, text, re.IGNORECASE)
        return urls

    def extract_certification_info(self, text: str) -> List[Dict[str, Any]]:
        """Extract certification information from text"""
        certifications = []
        urls = self.extract_urls(text)

        # Extract platform-specific certifications
        for platform, pattern in self.certification_patterns['platform_patterns'].items():
            matches = re.findall(pattern, text, re.IGNORECASE)
            for match in matches:
                full_url = f"https://{match}" if not match.startswith('http') else match
                certifications.append({
                    'platform': platform,
                    'verification_url': full_url,
                    'name': f"{platform} Certification",
                    'extraction_method': 'url_pattern',
                    'confidence': 0.9
                })

        # Extract general certification mentions
        lines = text.split('\n')
        for line in lines:
            line = line.strip()
            if any(keyword in line.lower() for keyword in ['certif', 'course', 'training', 'bootcamp']):
                # Try to extract certification name
                cert_info = self.parse_certification_line(line)
                if cert_info:
                    certifications.append(cert_info)

        # Match with Credly badges specifically
        credly_pattern = r'credly\.com/badges/([a-z0-9-]+)'
        credly_matches = re.findall(credly_pattern, text, re.IGNORECASE)
        for badge_id in credly_matches:
            certifications.append({
                'platform': 'Credly',
                'verification_url': f"https://www.credly.com/badges/{badge_id}",
                'badge_id': badge_id,
                'name': "Credly Badge",
                'extraction_method': 'credly_pattern',
                'confidence': 0.95
            })

        # Remove duplicates
        unique_certs = []
        seen_urls = set()
        for cert in certifications:
            url = cert.get('verification_url', '')
            if url not in seen_urls:
                seen_urls.add(url)
                unique_certs.append(cert)

        return unique_certs

    def parse_certification_line(self, line: str) -> Optional[Dict[str, Any]]:
        """Parse a single line to extract certification info"""
        # Look for common patterns
        patterns = [
            r'(.*?)(?:certification|certified|certificate)\s*(?:id|#)?\s*:?\s*([A-Z0-9\-]+)?',
            r'(.*?)(?:from|by)\s+([A-Za-z]+)',
            r'([A-Za-z].*?)(?:\s+|\s*-\s*)(https?://[^\s]+)'
        ]

        for pattern in patterns:
            match = re.search(pattern, line, re.IGNORECASE)
            if match:
                name = match.group(1).strip()
                if len(name) > 5:  # Minimum length for valid certification name
                    return {
                        'name': name,
                        'platform': 'Unknown',
                        'extraction_method': 'text_parsing',
                        'confidence': 0.7,
                        'raw_text': line
                    }
        return None

    def process_file(self, file_path: str) -> Dict[str, Any]:
        """Main processing method for the extraction agent"""
        # Extract text
        extracted_text = self.extract_text_from_file(file_path)

        # Extract certifications
        certifications = self.extract_certification_info(extracted_text)

        return {
            'extracted_text': extracted_text,
            'certifications': certifications,
            'extraction_stats': {
                'total_certifications': len(certifications),
                'text_length': len(extracted_text),
                'urls_found': len(self.extract_urls(extracted_text))
            }
        }

In [62]:
agent = CertificationExtractionAgent()
data = agent.extract_from_pdf('/content/sampleData.pdf')
print (data)

Certificate of Participation
This is to certify that
Ratheesh R
from SNS college of engineering as Team
The
Irregulars has participated in the
K! HACKS 2.0 - Idea
Submission of the
K! Hacks 2.0 organised by the
Anna
University .



In [53]:
# =============================================================================
# AGENT 2: VERIFICATION AGENT (RAG-ENABLED)
# =============================================================================

class VerificationAgent:
    """RAG-enabled agent for certificate verification"""

    def __init__(self):
      self.session = requests.Session()
      self.session.headers.update({
          'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36'
      })

      # Initialize vector database for RAG
      self.chroma_client = chromadb.Client()

      # Try to get existing collection or create new one
      try:
          self.collection = self.chroma_client.get_collection(name="verification_knowledge")
      except Exception:
          # Collection doesn't exist, create it
          self.collection = self.chroma_client.create_collection(
              name="verification_knowledge",
              metadata={"hnsw:space": "cosine"}
          )
          # Initialize with verification knowledge base only for new collections
          self.init_verification_knowledge()

      # Platform-specific verification methods
      self.verification_methods = {
          'Credly': self.verify_credly_badge,
          'Coursera': self.verify_coursera_certificate,
          'Google': self.verify_google_certificate,
          'AWS': self.verify_aws_certificate,
          'Microsoft': self.verify_microsoft_certificate,
          'LinkedIn': self.verify_linkedin_certificate,
          'HackerRank': self.verify_hackerrank_certificate,
          'FreeCodeCamp': self.verify_freecodecamp_certificate
      }

    def init_verification_knowledge(self):
        """Initialize the knowledge base for RAG"""
        verification_knowledge = [
            {
                "id": "credly_verification",
                "text": "Credly badges can be verified by accessing the badge URL directly. Valid badges show issuer information, issue date, and skills verified.",
                "platform": "Credly",
                "method": "url_verification"
            },
            {
                "id": "coursera_verification",
                "text": "Coursera certificates can be verified through coursera.org/verify/{certificate_id}. Valid certificates show course name, completion date, and grade.",
                "platform": "Coursera",
                "method": "url_verification"
            },
            {
                "id": "google_verification",
                "text": "Google certificates can be verified through Google Cloud certification registry or Coursera for Google Career Certificates.",
                "platform": "Google",
                "method": "registry_verification"
            },
            {
                "id": "aws_verification",
                "text": "AWS certifications can be verified through AWS Certification Validation portal using the validation number.",
                "platform": "AWS",
                "method": "validation_portal"
            }
        ]

        # Add to vector database
        for item in verification_knowledge:
            self.collection.add(
                documents=[item["text"]],
                metadatas=[{"platform": item["platform"], "method": item["method"]}],
                ids=[item["id"]]
            )

    def get_verification_context(self, platform: str, query: str) -> List[Dict]:
        """Retrieve relevant verification context using RAG"""
        results = self.collection.query(
            query_texts=[f"{platform} {query}"],
            n_results=3
        )

        context = []
        for i, doc in enumerate(results['documents'][0]):
            context.append({
                'document': doc,
                'metadata': results['metadatas'][0][i],
                'distance': results['distances'][0][i]
            })

        return context

    def verify_credly_badge(self, cert_info: Dict[str, Any]) -> Dict[str, Any]:
        """Verify Credly badge"""
        url = cert_info.get('verification_url', '')
        badge_id = cert_info.get('badge_id', '')

        try:
            response = self.session.get(url, timeout=10, verify=False)

            if response.status_code == 200:
                soup = BeautifulSoup(response.content, 'html.parser')

                # Check for badge verification indicators
                badge_title = soup.find('h1', class_='cr-badge-title')
                issuer_info = soup.find('div', class_='cr-badge-issuer')
                issue_date = soup.find('time', class_='cr-badge-issued-date')

                if badge_title and issuer_info:
                    return {
                        'status': 'verified',
                        'platform': 'Credly',
                        'badge_name': badge_title.get_text(strip=True) if badge_title else 'Unknown',
                        'issuer': issuer_info.get_text(strip=True) if issuer_info else 'Unknown',
                        'issue_date': issue_date.get_text(strip=True) if issue_date else 'Unknown',
                        'verification_method': 'credly_scraping',
                        'confidence': 0.9
                    }
                else:
                    return {
                        'status': 'suspicious',
                        'platform': 'Credly',
                        'message': 'Badge page exists but missing verification elements',
                        'confidence': 0.3
                    }
            else:
                return {
                    'status': 'invalid',
                    'platform': 'Credly',
                    'message': f'Badge page returned status {response.status_code}',
                    'confidence': 0.1
                }

        except Exception as e:
            return {
                'status': 'error',
                'platform': 'Credly',
                'message': f'Verification failed: {str(e)}',
                'confidence': 0.0
            }

    def verify_coursera_certificate(self, cert_info: Dict[str, Any]) -> Dict[str, Any]:
        """Verify Coursera certificate"""
        url = cert_info.get('verification_url', '')

        try:
            response = self.session.get(url, timeout=10, verify=False)

            if response.status_code == 200:
                content = response.text.lower()

                # Check for certificate validation indicators
                if 'certificate' in content and 'coursera' in content:
                    # Try to extract certificate details
                    soup = BeautifulSoup(response.content, 'html.parser')

                    return {
                        'status': 'verified',
                        'platform': 'Coursera',
                        'verification_method': 'url_verification',
                        'confidence': 0.8
                    }
                else:
                    return {
                        'status': 'suspicious',
                        'platform': 'Coursera',
                        'message': 'URL accessible but certificate validity unclear',
                        'confidence': 0.4
                    }
            else:
                return {
                    'status': 'invalid',
                    'platform': 'Coursera',
                    'message': f'Certificate URL returned status {response.status_code}',
                    'confidence': 0.1
                }

        except Exception as e:
            return {
                'status': 'error',
                'platform': 'Coursera',
                'message': f'Verification failed: {str(e)}',
                'confidence': 0.0
            }

    def verify_google_certificate(self, cert_info: Dict[str, Any]) -> Dict[str, Any]:
        """Verify Google certificate"""
        # Google certificates are often hosted on Coursera or other platforms
        return {
            'status': 'requires_manual_verification',
            'platform': 'Google',
            'message': 'Google certificates require manual verification through issuing platform',
            'confidence': 0.5
        }

    def verify_aws_certificate(self, cert_info: Dict[str, Any]) -> Dict[str, Any]:
        """Verify AWS certificate"""
        return {
            'status': 'requires_manual_verification',
            'platform': 'AWS',
            'message': 'AWS certificates require validation number for verification',
            'confidence': 0.5
        }

    def verify_microsoft_certificate(self, cert_info: Dict[str, Any]) -> Dict[str, Any]:
        """Verify Microsoft certificate"""
        return {
            'status': 'requires_manual_verification',
            'platform': 'Microsoft',
            'message': 'Microsoft certificates require manual verification through Microsoft Learn',
            'confidence': 0.5
        }

    def verify_linkedin_certificate(self, cert_info: Dict[str, Any]) -> Dict[str, Any]:
        """Verify LinkedIn certificate"""
        return {
            'status': 'requires_manual_verification',
            'platform': 'LinkedIn',
            'message': 'LinkedIn certificates require manual verification',
            'confidence': 0.5
        }

    def verify_hackerrank_certificate(self, cert_info: Dict[str, Any]) -> Dict[str, Any]:
        """Verify HackerRank certificate"""
        url = cert_info.get('verification_url', '')

        try:
            response = self.session.get(url, timeout=10, verify=False)

            if response.status_code == 200:
                return {
                    'status': 'verified',
                    'platform': 'HackerRank',
                    'verification_method': 'url_verification',
                    'confidence': 0.8
                }
            else:
                return {
                    'status': 'invalid',
                    'platform': 'HackerRank',
                    'message': f'Certificate URL returned status {response.status_code}',
                    'confidence': 0.1
                }

        except Exception as e:
            return {
                'status': 'error',
                'platform': 'HackerRank',
                'message': f'Verification failed: {str(e)}',
                'confidence': 0.0
            }

    def verify_freecodecamp_certificate(self, cert_info: Dict[str, Any]) -> Dict[str, Any]:
        """Verify FreeCodeCamp certificate"""
        url = cert_info.get('verification_url', '')

        try:
            response = self.session.get(url, timeout=10, verify=False)

            if response.status_code == 200:
                return {
                    'status': 'verified',
                    'platform': 'FreeCodeCamp',
                    'verification_method': 'url_verification',
                    'confidence': 0.8
                }
            else:
                return {
                    'status': 'invalid',
                    'platform': 'FreeCodeCamp',
                    'message': f'Certificate URL returned status {response.status_code}',
                    'confidence': 0.1
                }

        except Exception as e:
            return {
                'status': 'error',
                'platform': 'FreeCodeCamp',
                'message': f'Verification failed: {str(e)}',
                'confidence': 0.0
            }

    def verify_certificate(self, cert_info: Dict[str, Any]) -> Dict[str, Any]:
        """Main verification method"""
        platform = cert_info.get('platform', 'Unknown')

        # Get RAG context for verification
        context = self.get_verification_context(platform, "verification method")

        # Use platform-specific verification if available
        if platform in self.verification_methods:
            result = self.verification_methods[platform](cert_info)
        else:
            # Generic URL verification
            result = self.generic_url_verification(cert_info)

        # Add RAG context to result
        result['rag_context'] = context
        result['certificate_info'] = cert_info

        return result

    def generic_url_verification(self, cert_info: Dict[str, Any]) -> Dict[str, Any]:
        """Generic URL verification for unknown platforms"""
        url = cert_info.get('verification_url', '')

        if not url:
            return {
                'status': 'no_verification_url',
                'platform': cert_info.get('platform', 'Unknown'),
                'message': 'No verification URL provided',
                'confidence': 0.0
            }

        try:
            response = self.session.get(url, timeout=10, verify=False)

            if response.status_code == 200:
                return {
                    'status': 'url_accessible',
                    'platform': cert_info.get('platform', 'Unknown'),
                    'message': 'URL is accessible but verification method unknown',
                    'confidence': 0.6
                }
            else:
                return {
                    'status': 'invalid',
                    'platform': cert_info.get('platform', 'Unknown'),
                    'message': f'URL returned status {response.status_code}',
                    'confidence': 0.1
                }

        except Exception as e:
            return {
                'status': 'error',
                'platform': cert_info.get('platform', 'Unknown'),
                'message': f'Verification failed: {str(e)}',
                'confidence': 0.0
            }

    def verify_all_certificates(self, certifications: List[Dict[str, Any]]) -> List[Dict[str, Any]]:
        """Verify all certificates"""
        results = []

        for cert in certifications:
            result = self.verify_certificate(cert)
            results.append(result)

        return results


In [54]:
# =============================================================================
# AGENT 3: CREDIBILITY SCORING AGENT
# =============================================================================

class CredibilityScoringAgent:
    """Agent for calculating credibility scores"""

    def __init__(self):
        self.scoring_weights = {
            'verified': 100,
            'url_accessible': 60,
            'requires_manual_verification': 40,
            'suspicious': 20,
            'invalid': 0,
            'error': 0,
            'no_verification_url': 0
        }

        self.platform_weights = {
            'Credly': 1.0,
            'Coursera': 0.9,
            'Google': 0.95,
            'AWS': 0.95,
            'Microsoft': 0.95,
            'LinkedIn': 0.8,
            'HackerRank': 0.85,
            'FreeCodeCamp': 0.85,
            'Unknown': 0.5
        }

    def calculate_certificate_score(self, verification_result: Dict[str, Any]) -> Dict[str, Any]:
        """Calculate score for individual certificate"""
        status = verification_result.get('status', 'error')
        platform = verification_result.get('platform', 'Unknown')
        confidence = verification_result.get('confidence', 0.0)

        # Base score from status
        base_score = self.scoring_weights.get(status, 0)

        # Platform multiplier
        platform_multiplier = self.platform_weights.get(platform, 0.5)

        # Confidence multiplier
        confidence_multiplier = confidence

        # Final score calculation
        final_score = base_score * platform_multiplier * confidence_multiplier

        return {
            'certificate_score': round(final_score, 2),
            'base_score': base_score,
            'platform_multiplier': platform_multiplier,
            'confidence_multiplier': confidence_multiplier,
            'status': status,
            'platform': platform
        }

    def calculate_overall_credibility(self, verification_results: List[Dict[str, Any]]) -> Dict[str, Any]:
        """Calculate overall credibility score"""
        if not verification_results:
            return {
                'overall_score': 0.0,
                'grade': 'F',
                'total_certificates': 0,
                'verified_certificates': 0,
                'suspicious_certificates': 0,
                'invalid_certificates': 0,
                'certificate_scores': []
            }

        certificate_scores = []
        total_score = 0
        verified_count = 0
        suspicious_count = 0
        invalid_count = 0

        for result in verification_results:
            cert_score = self.calculate_certificate_score(result)
            certificate_scores.append(cert_score)
            total_score += cert_score['certificate_score']

            status = result.get('status', '')
            if status == 'verified':
                verified_count += 1
            elif status in ['suspicious', 'requires_manual_verification']:
                suspicious_count += 1
            elif status in ['invalid', 'error', 'no_verification_url']:
                invalid_count += 1

        # Calculate average score
        average_score = total_score / len(verification_results) if verification_results else 0

        # Determine grade
        grade = self.get_credibility_grade(average_score)

        return {
            'overall_score': round(average_score, 2),
            'grade': grade,
            'total_certificates': len(verification_results),
            'verified_certificates': verified_count,
            'suspicious_certificates': suspicious_count,
            'invalid_certificates': invalid_count,
            'certificate_scores': certificate_scores,
            'verification_rate': round(verified_count / len(verification_results) * 100, 1) if verification_results else 0
        }

    def get_credibility_grade(self, score: float) -> str:
        """Convert score to letter grade"""
        if score >= 90:
            return 'A+'
        elif score >= 80:
            return 'A'
        elif score >= 70:
            return 'B+'
        elif score >= 60:
            return 'B'
        elif score >= 50:
            return 'C+'
        elif score >= 40:
            return 'C'
        elif score >= 30:
            return 'D'
        else:
            return 'F'


In [55]:
# =============================================================================
# AGENT 4: FLAGGING & FEEDBACK AGENT
# =============================================================================

class FlaggingFeedbackAgent:
    """Agent for flagging issues and providing feedback"""

    def __init__(self):
        self.flag_criteria = {
            'expired': 'Certificate has expired',
            'invalid': 'Certificate URL is invalid or inaccessible',
            'suspicious': 'Certificate verification returned suspicious results',
            'no_verification_url': 'No verification URL provided',
            'error': 'Technical error occurred during verification',
            'requires_manual_verification': 'Certificate requires manual verification'
        }

    def flag_certificates(self, verification_results: List[Dict[str, Any]]) -> List[Dict[str, Any]]:
        """Flag problematic certificates"""
        flagged_items = []

        for result in verification_results:
            status = result.get('status', '')
            platform = result.get('platform', 'Unknown')
            cert_info = result.get('certificate_info', {})

            if status in self.flag_criteria:
                flag = {
                    'certificate_name': cert_info.get('name', 'Unknown Certificate'),
                    'platform': platform,
                    'flag_type': status,
                    'flag_reason': self.flag_criteria[status],
                    'message': result.get('message', ''),
                    'severity': self.get_flag_severity(status),
                    'verification_url': cert_info.get('verification_url', ''),
                    'recommendations': self.get_recommendations(status, platform)
                }
                flagged_items.append(flag)

        return flagged_items

    def get_flag_severity(self, status: str) -> str:
        """Determine flag severity"""
        high_severity = ['invalid', 'error']
        medium_severity = ['suspicious', 'expired']
        low_severity = ['requires_manual_verification', 'no_verification_url']

        if status in high_severity:
            return 'HIGH'
        elif status in medium_severity:
            return 'MEDIUM'
        elif status in low_severity:
            return 'LOW'
        else:
            return 'UNKNOWN'

    def get_recommendations(self, status: str, platform: str) -> List[str]:
        """Get recommendations based on flag type"""
        recommendations = []

        if status == 'invalid':
            recommendations.extend([
                "Verify the certificate URL is correct",
                "Check if the certificate has been moved to a different URL",
                "Contact the issuing organization for the correct verification link"
            ])
        elif status == 'suspicious':
            recommendations.extend([
                "Manually verify the certificate through the issuer's official website",
                "Check the certificate details for accuracy",
                "Consider removing if authenticity cannot be confirmed"
            ])
        elif status == 'expired':
            recommendations.extend([
                "Renew the certificate if possible",
                "Update your profile to reflect the expiration date",
                "Consider pursuing updated certification"
            ])
        elif status == 'no_verification_url':
            recommendations.extend([
                "Add a verification URL if available",
                "Include certificate ID or other identifying information",
                "Consider uploading a copy of the certificate"
            ])
        elif status == 'requires_manual_verification':
            recommendations.extend([
                f"Verify through {platform}'s official verification portal",
                "Provide additional verification details if available",
                "Consider contacting the issuer for verification assistance"
            ])
        elif status == 'error':
            recommendations.extend([
                "Try verification again later",
                "Check if the verification URL is accessible",
                "Report technical issues to the platform"
            ])

        return recommendations

    def generate_overall_recommendations(self, credibility_scores: Dict[str, Any],
                                       flagged_items: List[Dict[str, Any]]) -> List[str]:
        """Generate overall recommendations based on analysis"""
        recommendations = []

        overall_score = credibility_scores.get('overall_score', 0)
        verification_rate = credibility_scores.get('verification_rate', 0)
        total_certs = credibility_scores.get('total_certificates', 0)

        # Score-based recommendations
        if overall_score < 30:
            recommendations.append("🚨 CRITICAL: Most certificates could not be verified. Consider major profile cleanup.")
        elif overall_score < 50:
            recommendations.append("⚠️ WARNING: Low credibility score. Focus on verifiable certifications.")
        elif overall_score < 70:
            recommendations.append("📈 IMPROVEMENT NEEDED: Good foundation, but add more verifiable certifications.")
        elif overall_score < 90:
            recommendations.append("✅ GOOD: Strong credibility profile with minor improvements possible.")
        else:
            recommendations.append("🏆 EXCELLENT: Outstanding certification credibility!")

        # Verification rate recommendations
        if verification_rate < 50:
            recommendations.append("🔍 Add verification URLs or IDs to increase verification rate")

        # Certificate count recommendations
        if total_certs < 3:
            recommendations.append("📚 Consider adding more relevant certifications to strengthen your profile")
        elif total_certs > 15:
            recommendations.append("🎯 Focus on quality over quantity - highlight your most relevant certifications")

        # Flag-specific recommendations
        high_severity_flags = [flag for flag in flagged_items if flag.get('severity') == 'HIGH']
        if high_severity_flags:
            recommendations.append(f"🔥 URGENT: Address {len(high_severity_flags)} high-priority certification issues")

        # Platform diversity recommendations
        platforms = set(flag.get('platform', '') for flag in flagged_items)
        if len(platforms) > 5:
            recommendations.append("🌐 Great platform diversity in your certifications!")
        elif len(platforms) < 3:
            recommendations.append("🔄 Consider diversifying across different certification platforms")

        return recommendations


In [56]:
# =============================================================================
# LANGGRAPH WORKFLOW NODES
# =============================================================================

def extraction_node(state: VerificationState) -> VerificationState:
    """Node for certificate extraction"""
    agent = CertificationExtractionAgent()
    result = agent.process_file(state['input_file_path'])

    return {
        **state,
        'extracted_text': result['extracted_text'],
        'certifications': result['certifications']
    }

def verification_node(state: VerificationState) -> VerificationState:
    """Node for certificate verification"""
    agent = VerificationAgent()
    results = agent.verify_all_certificates(state['certifications'])

    return {
        **state,
        'verification_results': results
    }

def scoring_node(state: VerificationState) -> VerificationState:
    """Node for credibility scoring"""
    agent = CredibilityScoringAgent()
    scores = agent.calculate_overall_credibility(state['verification_results'])

    return {
        **state,
        'credibility_scores': scores
    }

def flagging_node(state: VerificationState) -> VerificationState:
    """Node for flagging and feedback"""
    agent = FlaggingFeedbackAgent()
    flagged_items = agent.flag_certificates(state['verification_results'])
    recommendations = agent.generate_overall_recommendations(
        state['credibility_scores'], flagged_items
    )

    return {
        **state,
        'flagged_items': flagged_items,
        'recommendations': recommendations
    }

def report_generation_node(state: VerificationState) -> VerificationState:
    """Node for final report generation"""
    final_report = {
        'timestamp': datetime.now().isoformat(),
        'summary': {
            'overall_score': state['credibility_scores'].get('overall_score', 0),
            'grade': state['credibility_scores'].get('grade', 'F'),
            'total_certificates': len(state['certifications']),
            'verified_certificates': state['credibility_scores'].get('verified_certificates', 0),
            'flagged_certificates': len(state['flagged_items'])
        },
        'detailed_results': state['verification_results'],
        'credibility_analysis': state['credibility_scores'],
        'flags_and_issues': state['flagged_items'],
        'recommendations': state['recommendations']
    }

    return {
        **state,
        'final_report': final_report
    }


In [57]:
# =============================================================================
# BUILD LANGGRAPH WORKFLOW
# =============================================================================

def build_verification_workflow() -> CompiledGraph:
    """Build the certificate verification workflow"""
    builder = StateGraph(VerificationState)

    # Add nodes
    builder.add_node("extract", extraction_node)
    builder.add_node("verify", verification_node)
    builder.add_node("score", scoring_node)
    builder.add_node("flag", flagging_node)
    builder.add_node("report", report_generation_node)

    # Define workflow
    builder.set_entry_point("extract")
    builder.add_edge("extract", "verify")
    builder.add_edge("verify", "score")
    builder.add_edge("score", "flag")
    builder.add_edge("flag", "report")
    builder.add_edge("report", END)

    return builder.compile()


In [58]:
# =============================================================================
# GRADIO INTERFACE
# =============================================================================

def process_certificate_verification(file):
    """Main processing function for Gradio interface"""
    if file is None:
        return "No file uploaded", "", "", ""

    try:
        # Initialize workflow
        workflow = build_verification_workflow()

        # Save uploaded file temporarily
        temp_file_path = file.name

        # Run verification workflow
        initial_state = {
            'input_file_path': temp_file_path,
            'extracted_text': '',
            'certifications': [],
            'verification_results': [],
            'credibility_scores': {},
            'flagged_items': [],
            'recommendations': [],
            'final_report': {}
        }

        result = workflow.invoke(initial_state)

        # Format results for display
        summary_text = format_summary_report(result['final_report'])
        detailed_text = format_detailed_report(result['final_report'])
        recommendations_text = format_recommendations(result['recommendations'])
        flags_text = format_flags(result['flagged_items'])

        return summary_text, detailed_text, recommendations_text, flags_text

    except Exception as e:
        error_msg = f"Error processing file: {str(e)}"
        return error_msg, "", "", ""

def format_summary_report(report: Dict[str, Any]) -> str:
    """Format summary report for display"""
    summary = report.get('summary', {})

    text = f"""
# 🏆 CERTIFICATE VERIFICATION SUMMARY

## Overall Results
- **Credibility Score**: {summary.get('overall_score', 0)}/100
- **Grade**: {summary.get('grade', 'F')}
- **Total Certificates**: {summary.get('total_certificates', 0)}
- **Verified Certificates**: {summary.get('verified_certificates', 0)}
- **Flagged Issues**: {summary.get('flagged_certificates', 0)}

## Verification Rate
{summary.get('verified_certificates', 0)}/{summary.get('total_certificates', 0)} certificates successfully verified
({round(summary.get('verified_certificates', 0)/max(summary.get('total_certificates', 1), 1)*100, 1)}%)
"""
    return text

def format_detailed_report(report: Dict[str, Any]) -> str:
    """Format detailed report for display"""
    results = report.get('detailed_results', [])

    text = "# 📋 DETAILED VERIFICATION RESULTS\n\n"

    for i, result in enumerate(results, 1):
        cert_info = result.get('certificate_info', {})
        text += f"## Certificate {i}: {cert_info.get('name', 'Unknown')}\n"
        text += f"- **Platform**: {result.get('platform', 'Unknown')}\n"
        text += f"- **Status**: {result.get('status', 'Unknown').upper()}\n"
        text += f"- **Confidence**: {result.get('confidence', 0)*100:.1f}%\n"

        if result.get('verification_url'):
            text += f"- **URL**: {result.get('verification_url')}\n"

        if result.get('message'):
            text += f"- **Message**: {result.get('message')}\n"

        text += "\n---\n\n"

    return text

def format_recommendations(recommendations: List[str]) -> str:
    """Format recommendations for display"""
    if not recommendations:
        return "No specific recommendations available."

    text = "# 💡 RECOMMENDATIONS\n\n"
    for i, rec in enumerate(recommendations, 1):
        text += f"{i}. {rec}\n\n"

    return text

def format_flags(flagged_items: List[Dict[str, Any]]) -> str:
    """Format flags for display"""
    if not flagged_items:
        return "# ✅ NO ISSUES FOUND\n\nAll certificates passed verification checks!"

    text = "# 🚩 FLAGGED ISSUES\n\n"

    for flag in flagged_items:
        severity_emoji = {"HIGH": "🔥", "MEDIUM": "⚠️", "LOW": "ℹ️"}.get(flag.get('severity', 'LOW'), "ℹ️")

        text += f"## {severity_emoji} {flag.get('certificate_name', 'Unknown Certificate')}\n"
        text += f"- **Platform**: {flag.get('platform', 'Unknown')}\n"
        text += f"- **Issue**: {flag.get('flag_reason', 'Unknown issue')}\n"
        text += f"- **Severity**: {flag.get('severity', 'Unknown')}\n"

        if flag.get('message'):
            text += f"- **Details**: {flag.get('message')}\n"

        recommendations = flag.get('recommendations', [])
        if recommendations:
            text += "- **Recommendations**:\n"
            for rec in recommendations:
                text += f"  - {rec}\n"

        text += "\n---\n\n"

    return text


In [59]:
# =============================================================================
# GRADIO APP
# =============================================================================

def create_gradio_interface():
    """Create Gradio interface"""

    with gr.Blocks(title="AI Certificate Verification System", theme=gr.themes.Soft()) as app:
        gr.Markdown("""
        # 🎓 AI Certificate Verification System

        Upload your resume, portfolio, or LinkedIn profile to verify the authenticity of your certifications.
        Our AI agents will extract, verify, and analyze your certificates to provide a comprehensive credibility report.

        **Supported formats**: PDF, Images (JPG, PNG, etc.), Text files
        """)

        with gr.Row():
            with gr.Column(scale=1):
                file_input = gr.File(
                    label="📁 Upload Document",
                    file_types=[".pdf", ".jpg", ".jpeg", ".png", ".txt", ".tiff", ".bmp"],
                    file_count="single"
                )

                process_btn = gr.Button("🔍 Verify Certificates", variant="primary", size="lg")

        with gr.Row():
            with gr.Column():
                summary_output = gr.Markdown(label="📊 Summary Report")

            with gr.Column():
                recommendations_output = gr.Markdown(label="💡 Recommendations")

        with gr.Row():
            with gr.Column():
                detailed_output = gr.Markdown(label="📋 Detailed Results")

            with gr.Column():
                flags_output = gr.Markdown(label="🚩 Issues & Flags")

        process_btn.click(
            fn=process_certificate_verification,
            inputs=[file_input],
            outputs=[summary_output, detailed_output, recommendations_output, flags_output]
        )

        gr.Markdown("""
        ---
        ## 🔍 How It Works

        1. **Extraction Agent**: Parses your document to find certifications and their details
        2. **Verification Agent**: Uses RAG-enabled verification to check certificate authenticity
        3. **Scoring Agent**: Calculates credibility scores based on verification results
        4. **Flagging Agent**: Identifies issues and provides actionable recommendations

        ## 🌟 Features

        - ✅ Multi-platform verification (Credly, Coursera, Google, AWS, etc.)
        - 🤖 AI-powered certificate extraction
        - 🔗 URL validation and web scraping
        - 📊 Comprehensive credibility scoring
        - 🚩 Issue flagging and recommendations
        - 📱 Support for multiple file formats
        """)

    return app


In [60]:
# =============================================================================
# MAIN EXECUTION
# =============================================================================

if __name__ == "__main__":
    # Create and launch Gradio interface
    app = create_gradio_interface()
    app.launch(
        share=True,
        server_name="0.0.0.0",
        # server_port=7860, # Remove fixed port to allow Gradio to find an available one
        show_error=True
    )

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://a6c20936f53ba4a904.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
